# ChromaDB Embeddings Setup and Inspection

## 1. Initial ChromaDB Setup, Embedding Generation, and Verification (Method 1)

This section demonstrates the initial setup of a ChromaDB collection, generation of embeddings from a JSON file, and verification of the stored data. It creates a persistent ChromaDB instance, loads text data, generates embeddings using a default embedding function, and stores them with associated metadata and unique IDs. Finally, it retrieves and prints a sample of the stored embeddings to confirm the process.

In [6]:
import json
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. Setup paths and context-specific names
file_name = "chapter7_72_thinkandact_output.json"
context_name = "chapter7_72_thinkandact"

# Structure: project_root -> data_embedding -> chapter_db
base_dir = "data_embedding"
db_folder_name = f"{context_name}_db"
db_path = os.path.join(base_dir, db_folder_name)

# Ensure the directory exists
os.makedirs(db_path, exist_ok=True)

# 2. Load the JSON data
with open(file_name, 'r') as f:
    json_data = json.load(f)

# 3. Initialize Chroma Client for this specific directory
client = chromadb.PersistentClient(path=db_path)

# 4. Define the Embedding Function
default_ef = embedding_functions.DefaultEmbeddingFunction()

# 5. Create a collection specific to this chapter
# The variable name is kept specific to your context
chapter7_72_thinkandact_collection = client.get_or_create_collection(
    name=f"{context_name}_embeddings",
    embedding_function=default_ef
)

# 6. Prepare data for storage
documents = [item["thinkandact"] for item in json_data]
metadatas = [
    {
        "chapter": item["chapter"],
        "pg_no": item["pg no"],
        "source_file": file_name
    } for item in json_data
]
ids = [f"{context_name}_id_{i}" for i in range(len(json_data))]

# 7. Generate and store embeddings
chapter7_72_thinkandact_collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

# 8. Retrieve and store the result in your requested variable
chapter7_72_thinkandact_output = chapter7_72_thinkandact_collection.get(
    include=['embeddings', 'documents', 'metadatas']
)

# 9. Verify and Print (handling the NumPy array ambiguity)
raw_embeddings = chapter7_72_thinkandact_output.get('embeddings')

if raw_embeddings is not None and len(raw_embeddings) > 0:
    print(f"--- Process Complete ---")
    print(f"Database Path: {db_path}")
    print(f"Collection Name: {chapter7_72_thinkandact_collection.name}")
    print(f"First Embedding Sample (10 dims): {raw_embeddings[0][:10]}...")
    print(f"Total Vectors Stored: {len(raw_embeddings)}")
else:
    print("Failed to retrieve embeddings.")

--- Process Complete ---
Database Path: data_embedding/chapter7_72_thinkandact_db
Collection Name: chapter7_72_thinkandact_embeddings
First Embedding Sample (10 dims): [ 0.00818573 -0.0441445   0.04858383  0.0667825   0.01142112  0.01206521
 -0.03842649 -0.03420644  0.05174617  0.04097006]...
Total Vectors Stored: 1


## 2. Refined ChromaDB Setup with Source Snapshot (Method 2)

This section provides an alternative, and potentially more robust, way to set up the ChromaDB. It includes an `!pip install chromadb` command to ensure the library is available. Similar to the previous section, it initializes ChromaDB, prepares data for storage, and generates embeddings. A key addition here is the creation of a `source_snapshot.json` file within the database directory, which serves as a record of the original data used to populate the embeddings.

In [7]:
import json
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. Setup paths
file_name = "chapter7_72_thinkandact_output.json"
context_name = "chapter7_72_thinkandact"

# This matches: project_root -> data_embedding -> chapter_db
base_dir = "data_embedding"
db_folder_name = f"{context_name}_db"
db_path = os.path.join(base_dir, db_folder_name)

os.makedirs(db_path, exist_ok=True)

# 2. Load the JSON data
with open(file_name, 'r') as f:
    json_data = json.load(f)

# 3. Initialize Chroma (Creates the sqlite3 and uuid folders inside db_path)
client = chromadb.PersistentClient(path=db_path)
default_ef = embedding_functions.DefaultEmbeddingFunction()

# 4. Collection setup
chapter7_72_thinkandact_collection = client.get_or_create_collection(
    name=f"{context_name}_embeddings",
    embedding_function=default_ef
)

# 5. Data Preparation
documents = [item["thinkandact"] for item in json_data]
metadatas = [{"chapter": item["chapter"], "pg_no": item["pg no"]} for item in json_data]
ids = [f"{context_name}_id_{i}" for i in range(len(json_data))]

# 6. Generate and Store
chapter7_72_thinkandact_collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

# 7. ADDED: Create the 'source_snapshot.json' in the same folder
snapshot_path = os.path.join(db_path, "source_snapshot.json")
with open(snapshot_path, 'w') as f:
    json.dump(json_data, f, indent=4)

# 8. Result variable
chapter7_72_thinkandact_output = chapter7_72_thinkandact_collection.get(include=['embeddings'])

print(f"Structure verified. Files created in: {db_path}")

Structure verified. Files created in: data_embedding/chapter7_72_thinkandact_db


## 3. Inspecting Stored Embeddings

This final section focuses on retrieving and inspecting the data that has been stored in the ChromaDB collection. It fetches the IDs, original documents (text), and generated embeddings. The code then iterates through these results, printing a formatted overview of each entry, including its ID, a snippet of the original text, and a sample of its corresponding embedding vector. This helps verify that the data was correctly stored and can be retrieved as expected.

In [8]:
# 1. Retrieve the data with all components included
# We explicitly ask for 'embeddings' because they are excluded by default
inspection_data = chapter7_72_thinkandact_collection.get(
    include=['embeddings', 'documents', 'metadatas']
)

# 2. Iterate through the results to check the mapping
print(f"{'ID':<15} | {'TEXT PREVIEW':<40} | {'VECTOR SAMPLE'}")
print("-" * 80)

for i in range(len(inspection_data['ids'])):
    id_val = inspection_data['ids'][i]

    # Get a snippet of the text
    text_snippet = inspection_data['documents'][i][:37] + "..."

    # Get the first few dimensions of the embedding
    vector = inspection_data['embeddings'][i]
    vector_snippet = f"[{vector[0]:.4f}, {vector[1]:.4f}, {vector[2]:.4f}...]"

    print(f"{id_val:<15} | {text_snippet:<40} | {vector_snippet}")

# 3. Print the total count for verification
print("-" * 80)
print(f"Total entries in embedding variable: {len(inspection_data['ids'])}")

ID              | TEXT PREVIEW                             | VECTOR SAMPLE
--------------------------------------------------------------------------------
chapter7_72_thinkandact_id_0 | We sometimes are endangered by the mo... | [0.0082, -0.0441, 0.0486...]
--------------------------------------------------------------------------------
Total entries in embedding variable: 1
